
###INCREMENTAL DATA LOADING


Creating Database

In [0]:
CREATE DATABASE sales;


Create a Table 

In [0]:
CREATE OR REPLACE TABLE sales.Orders (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
);

Inserting data into database

In [0]:
INSERT INTO sales.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00),
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00),
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00),
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00),
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);


In [0]:
select * from sales.orders


#DATA WAREHOUSING

In [0]:
create database salesDWH;


## simple explanation for understanding 

###Creating the STAGING Layer 

In [0]:
-- initial load 
create table salesDWH.stg_sales 
AS 
SELECT * FROM sales.Orders

In [0]:
select * from salesDWH.stg_sales  -- data exactly matching with source

### Transformation

we can store the Query by creating a VIEW 

In [0]:
CREATE or replace VIEW salesDWH.trans_sales
AS
select * from salesDWH.stg_sales where Quantity is not null


###CORE LAYER

In [0]:
CREATE OR REPLACE TABLE salesDWH.core_sales (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
);

In [0]:
INSERT INTO salesDWH.core_sales
select * from salesDWH.trans_sales;

datawarehouse core layer data viewing 

In [0]:
select * from salesDWH.core_sales 


Inserting more data to sales.Orders table(source table)

In [0]:
INSERT INTO sales.Orders
(OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES
(11, '2024-02-11', 108, 'Hannah Green', 'hannah@example.com', 210, 'Printer', 'Accessories', 302, 'Europe', 'Spain', 2, 180.00, 360.00),

(12, '2024-02-12', 105, 'Eve Adams', 'eve@example.com', 211, 'External SSD', 'Storage', 301, 'North America', 'Canada', 1, 220.00, 220.00),

(13, '2024-02-13', 109, 'Ian Clark', 'ian@example.com', 202, 'Smartphone', 'Electronics', 303, 'Asia', 'Singapore', 2, 500.00, 1000.00),

(14, '2024-02-14', 110, 'Julia Turner', 'julia@example.com', 212, 'Wireless Router', 'Networking', 301, 'North America', 'Mexico', 3, 120.00, 360.00),

(15, '2024-02-15', 103, 'Charlie Brown', 'charlie@example.com', 213, 'Webcam', 'Accessories', 302, 'Europe', 'Netherlands', 2, 90.00, 180.00);

In [0]:
select * from sales.orders

### Incremental Load 

-means process only the new data that came after the last load 
we do that by comparing based on the **CDC column** value 



### Staging Layer 


- **CREATE OR REPLACE TABLE** truncates the data of staging table 
- loads the data from the source table that is greater than the CDC value ****

In [0]:
CREATE OR REPLACE TABLE salesDWH.stg_sales 
as
select * from sales.orders
where OrderDate >'2024-02-10'


### Transformation

- CREATE OR REPLACE VIEW  now create the view based on the data that is stored in the staging layer 

In [0]:
CREATE OR REPLACE VIEW salesDWH.trans_sales
AS
select * from salesDWH.stg_sales where Quantity is not null

In [0]:
select * from salesDWH.trans_sales

--we'll see that the view salesDWH.trans_sales now only have the new records as the staging table contains only the new records 


now we just can't give create or replace table in the CORE layer as if we do that it will delete the data already present in the core layer table 


### Inserting the updated records to the CORE_Layer

In [0]:
INSERT into salesdwh.core_sales
select * from salesDWH.trans_sales

In [0]:
select * from salesdwh.core_sales


# We don't do this mannually IT has to be done by DATA MODELS

In [0]:
create database new_sales;

In [0]:
CREATE OR REPLACE TABLE new_sales.Orders (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
);

In [0]:
INSERT INTO new_sales.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00),
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00),
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00),
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00),
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);


In [0]:
select * from new_sales.orders

create orders warehouse

In [0]:
Create database OrdersDWH;


###Staging Layer

In [0]:
create or replace table OrdersDWH.stg_sales
as 
select * from new_sales.orders

###Transformation

In [0]:
Create or replace view ordersdwh.trans_sales
as 
select * from OrdersDWH.stg_sales
where Quantity is not null 

In [0]:
select * from ordersdwh.trans_sales


# Creating Dimension Tables

### Dim_Customers

In [0]:
select distinct(CustomerID) as CustomerID,
CustomerName,
CustomerEmail
from  ordersdwh.trans_sales



### We use Surrogate Keys for the DIMENSION Table (instead of it's rows) in the FACT Table 


### Generating Surrogate Key using row_number()

In [0]:
--using subquery

select t.*,row_number() OVER (ORDER BY t.CustomerID) as DimCustomerKey
from (
select distinct CustomerID,
--row_number() OVER (ORDER BY CustomerID) as DimCustomerKey,
CustomerName,
CustomerEmail
from  ordersdwh.trans_sales) AS t


In [0]:
--using CTE

with distinct_customers as (
  select distinct CustomerID,
    CustomerName,
    CustomerEmail
  from ordersdwh.trans_sales
)
select t.*, row_number() OVER (ORDER BY t.CustomerID) as DimCustomerKey
from distinct_customers t

In [0]:

-- this is not giving us the distinct value so we  need to use Subquery of CTE as shown in the above sell 

select distinct CustomerID,
row_number() OVER (ORDER BY CustomerID) as DimCustomerKey,
CustomerName,
CustomerEmail
from  ordersdwh.trans_sales

In [0]:
create or replace table ordersdwh.DimCustomers(
    CustomerID int,
    CustomerName varchar(100),
    CustomerEmail varchar(100),
    DimCustomerKey int
)

In [0]:
create or replace view  ordersdwh.view_DimCustomers
as
select 
distinct(CustomerID) as CustomerID,
row_number() OVER (ORDER BY CustomerID) as DimCustomerKey,
CustomerName,
CustomerEmail
from (
select t.*,row_number() OVER (ORDER BY t.CustomerID) as DimCustomerKey
from (
select distinct CustomerID,
--row_number() OVER (ORDER BY CustomerID) as DimCustomerKey,
CustomerName,
CustomerEmail
from  ordersdwh.trans_sales) AS t)

In [0]:
create or replace view  ordersdwh.view_DimCustomers
as
with distinct_customers as (
  select distinct CustomerID,
    CustomerName,
    CustomerEmail
  from ordersdwh.trans_sales
)
select t.*, row_number() OVER (ORDER BY t.CustomerID) as DimCustomerKey
from distinct_customers t

In [0]:
select * from ordersdwh.view_DimCustomers

In [0]:
describe ordersdwh.DimCustomers

In [0]:
select count(*) from ordersdwh.DimCustomers

In [0]:
insert into ordersdwh.DimCustomers
select * from ordersdwh.view_DimCustomers

### DimProduct

In [0]:
create or replace table ordersdwh.DimProduct(
    ProductID int,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(100),
    DimProductKey int
    )

In [0]:
create or replace View ordersdwh.view_DimProduct
as 
with distinct_products as (
    select  distinct ProductID,
    ProductName,
    ProductCategory
    from ordersdwh.trans_sales
)
select t.*, row_number() OVER (ORDER BY t.ProductID) as DimProductKey
from distinct_products t

In [0]:
select * from ordersdwh.view_DimProduct

In [0]:
insert into ordersdwh.DimProduct
select * from ordersdwh.view_DimProduct

### DimRegion

In [0]:
create or replace table ordersdwh.DimRegion(
    RegionID int,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    DimRegionKey int
    )

In [0]:
create or replace view ordersdwh.view_DimRegion
as 
with distinct_region as (
    select  distinct RegionID,
    RegionName,
    ProductCategory
    from ordersdwh.trans_sales
)
select t.*, row_number() OVER (ORDER BY t.RegionID) as DimRegionKey
from distinct_region t

In [0]:
select * from ordersdwh.view_DimRegion

In [0]:
insert into ordersdwh.dimregion
select * from ordersdwh.view_DimRegion

In [0]:
select * from ordersdwh.dimregion

### DimDate 

In [0]:
create or replace table ordersdwh.DimDate(
OrderDate Date,
DimDateKey int
    )

In [0]:
create or replace view ordersdwh.view_DimDate
as 
with distinct_date as (
    select  distinct OrderDate
    from ordersdwh.trans_sales
)
select t.*, row_number() OVER (ORDER BY t.OrderDate) as DimDateKey
from distinct_date t

In [0]:
insert into ordersdwh.dimdate
select * from ordersdwh.view_DimDate


## FACT TABLE

In [0]:
-- query
select  t.OrderID,
c.DimCustomerKey,
p.DimProductKey,
r.DimRegionKey,
d.DimDateKey,
t.Quantity,
t.UnitPrice,
t.TotalAmount
from ordersdwh.trans_sales t
LEFT JOIN ordersdwh.dimCustomers c on t.CustomerID=c.CustomerID
LEFT JOIN ordersdwh.dimproduct p on t.ProductID=p.ProductID
LEFT JOIN ordersdwh.dimregion r on t.RegionID=r.RegionID
LEFT JOIN ordersdwh.dimdate d on t.OrderDate=d.OrderDate




In [0]:
CREATE or replace table ordersdwh.FactSales(
    OrderID int,
    Quantity    decimal,
    UnitPrice   decimal,
    TotalAmount decimal,
    DimCustomerKey int,
    DimProductKey int,
    DimRegionKey int,
    DimDateKey int 
)

In [0]:
select 
F.OrderID,
F.Quantity,
F.UnitPrice,
F.TotalAmount,
c.DimCustomerKey,
p.DimProductKey,
r.DimRegionKey,
d.DimDateKey
from ordersdwh.trans_sales F
LEFT JOIN ordersdwh.dimCustomers c on F.CustomerID=c.CustomerID
LEFT JOIN ordersdwh.dimproduct p on F.ProductID=p.ProductID
LEFT JOIN ordersdwh.dimregion r on r.RegionID=F.RegionID
LEFT JOIN ordersdwh.dimdate d on F.OrderDate=d.OrderDate

In [0]:
INSERT INTO ordersdwh.factsales(
    OrderID,
    Quantity,
    UnitPrice,
    TotalAmount,
    DimCustomerKey,
    DimProductKey,
    DimRegionKey,
    DimDateKey
)
select 
F.OrderID,
F.Quantity,
F.UnitPrice,
F.TotalAmount,
c.DimCustomerKey,
p.DimProductKey,
r.DimRegionKey,
d.DimDateKey
from ordersdwh.trans_sales F
LEFT JOIN ordersdwh.dimCustomers c on F.CustomerID=c.CustomerID
LEFT JOIN ordersdwh.dimproduct p on F.ProductID=p.ProductID
LEFT JOIN ordersdwh.dimregion r on r.RegionID=F.RegionID
LEFT JOIN ordersdwh.dimdate d on F.OrderDate=d.OrderDate

In [0]:
select * from ordersdwh.factsales


# Slowly changing Dimensions

In [0]:
Create database sales_scd;

In [0]:
CREATE OR REPLACE TABLE sales_scd.Orders (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
);


In [0]:
INSERT INTO sales_scd.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00),
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00),
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00),
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00),
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);



### SCD Type -1

In [0]:
select * from sales_scd.Orders

**STEP 1- Create a view on top of the dimension**

In [0]:
create or replace view sales_scd.view_dimProducts
as
select distinct ProductID as ProductID ,ProductName, ProductCategory from sales_scd.Orders

**STEP 2-CREATE A DIM TABLE**

In [0]:
create or replace table sales_scd.dimproduct(
    ProductID INT,
    ProductName VARCHAR(100), 
    ProductCategory VARCHAR(100)
)

In [0]:
INSERT into sales_scd.dimproduct
select * from sales_scd.view_dimProducts

In [0]:
select * from sales_scd.dimproduct

In [0]:
INSERT INTO sales_scd.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(1, '2024-02-11', 101, 'Alice Johnson', 'alice@example.com', 201, 'Gaming Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-12', 102, 'Bob Smith', 'bob@example.com', 230, 'Airpods', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00)

In [0]:
select * from sales_scd.dimproduct


changes the view as we are only want to process the incremental values 

In [0]:
create or replace view sales_scd.view_dimProducts
as
select distinct ProductID as ProductID ,ProductName, ProductCategory from sales_scd.Orders where OrderDate>'2024-02-10'

In [0]:
select * from sales_scd.view_dimProducts



### MERGE SCD Type-1

In [0]:
MERGE INTO sales_scd.dimproduct as trg
USING sales_scd.view_dimProducts as src
ON trg.ProductID=src.ProductID
when MATCHED THEN UPDATE SET *  -- all the columns must be updated 
when NOT MATCHED THEN INSERT *